# Paragraph splitting — first look

**Goal:** chunks (2-3k tokens, sized for tree traversal) are too big to auto-inject into an
agent's context for RAG; we want ~100-200-token **paragraphs** as the injection granularity.
Before choosing a splitting approach, see what the *existing* recursive-delimiter splitter
produces when simply given a paragraph-scale size target.

**Method:** `paragraphing.toml` — identical separator hierarchies as `chunking.toml`, size
target 0.75 KB (code/docs) / 0.5 KB (data). The splitter functions below are copied verbatim
from `research.ipynb` (research history stays untouched; extract into a shared module only
once paragraphing survives eyeball review). Sample: per category, the 2 largest chunks (worst
case — most internal structure to cut) + 2 seeded-random ones. Every paragraph is printed in
full with its boundaries so the split quality can be judged by eye.

**This notebook decides nothing** — it produces the evidence for the
what-is-a-good-paragraph-boundary discussion (AST/tree-sitter vs delimiter hierarchy).

In [1]:
import random
import sqlite3
import tomllib
from collections import defaultdict
from pathlib import Path

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"

with open(APP_DIR / "paragraphing.toml", "rb") as f:
    PARA_CFG = tomllib.load(f)["categories"]

SEED = 42
PER_CATEGORY_LARGEST = 2
PER_CATEGORY_RANDOM = 2

In [2]:
# --- splitter: copied VERBATIM from research.ipynb (do not edit here; a shared module
# --- happens only after the paragraphing approach is settled) ---

def split_text(text, separators, chunk_size, base_offset=0):
    """Recursively split on the first separator, falling back to the next
    separator (then raw chars) for any piece still over chunk_size."""
    if len(text) <= chunk_size or not separators:
        return [(text, False, base_offset, base_offset + len(text))] if text else []
    sep, *rest = separators
    if not sep:
        return [(ch, True, base_offset + i, base_offset + i + 1) for i, ch in enumerate(text)]
    result = []
    pos = base_offset
    parts = text.split(sep)
    for idx, piece in enumerate(parts):
        start, end = pos, pos + len(piece)
        if len(piece) > chunk_size:
            result.extend(split_text(piece, rest, chunk_size, start))
        elif piece:
            result.append((piece, False, start, end))
        pos = end + (len(sep) if idx < len(parts) - 1 else 0)
    return result


def merge_pieces(pieces, chunk_size):
    """Pack pieces up to chunk_size; raw-char pieces join with nothing, real
    pieces with a newline. A cut only ever falls between two whole pieces."""
    chunks = []
    current, current_start, current_end, current_ends_raw = "", None, None, False
    for piece, is_raw, start, end in pieces:
        join = "" if (is_raw and current_ends_raw) else "\n"
        candidate = f"{current}{join}{piece}" if current else piece
        if len(candidate) <= chunk_size:
            current = candidate
            current_start = start if current_start is None else current_start
            current_end = end
        else:
            if current:
                chunks.append((current, current_start, current_end))
            current, current_start, current_end = piece, start, end
        current_ends_raw = is_raw
    if current:
        chunks.append((current, current_start, current_end))
    return chunks


def split_into_paragraphs(chunk_text, category):
    cfg = PARA_CFG[category]
    size = int(cfg["paragraph_size_kb"] * 1024)
    pieces = split_text(chunk_text, cfg["separators"], size)
    return merge_pieces(pieces, size)

## Sample selection

Per category: the 2 largest chunks + 2 seeded-random ones. Largest chunks are the honest
test — a chunk near the 10 KB cap has the most internal structure a bad splitter can cut
through; tiny chunks would pass trivially by needing no split at all.

In [3]:
db = sqlite3.connect(DB_PATH)
db.row_factory = sqlite3.Row
rows = [dict(r) for r in db.execute(
    "SELECT uri, instance, path, category, byte_start, byte_end, hash FROM chunks")]
db.close()

for r in rows:
    raw = (GIT_CACHE / r["instance"] / r["path"]).read_bytes()[r["byte_start"]:r["byte_end"]]
    r["content"] = raw.decode("utf-8", errors="replace")

rng = random.Random(SEED)
by_cat = defaultdict(list)
for r in rows:
    by_cat[r["category"]].append(r)

samples = []
for cat, cat_rows in sorted(by_cat.items()):
    largest = sorted(cat_rows, key=lambda r: -len(r["content"]))[:PER_CATEGORY_LARGEST]
    rest = [r for r in cat_rows if r not in largest]
    samples += largest + rng.sample(rest, min(PER_CATEGORY_RANDOM, len(rest)))

print(f"{len(rows)} chunks total; sampled {len(samples)}:")
for r in samples:
    print(f"  [{r['category']:>4}] {len(r['content']):>6} chars  {r['path']}")

510 chunks total; sampled 12:
  [code]  10240 chars  branow/webphone/server/src/test/java/com/scisbo/webphone/services/ContactServiceTest.java
  [code]  10216 chars  branow/webphone/server/src/main/java/com/scisbo/webphone/services/ContactService.java
  [code]  10111 chars  branow/webphone/server/src/test/java/com/scisbo/webphone/controllers/ContactControllerTest.java
  [code]   2773 chars  branow/webphone/client/src/pages/calls/CallPreviewControls.tsx
  [data]   2046 chars  branow/webphone/client/public/locales/uk/translation.json
  [data]   2043 chars  branow/webphone/client/public/locales/en/translation.json
  [data]   2007 chars  branow/webphone/server/pom.xml
  [data]   2041 chars  branow/webphone/server/src/test/resources/com/scisbo/webphone/common/data/accounts.json
  [docs]  10128 chars  branow/webphone/LICENSE
  [docs]  10083 chars  branow/webphone/server/mvnw
  [docs]     68 chars  branow/webphone/client/src/index.css
  [docs]    135 chars  branow/webphone/client/public/silen

## Split output — every paragraph in full, for eyeball review

Per paragraph: index, char length, ~token estimate (chars/4). Boundary markers are the
thing to judge: does each paragraph read as a self-contained unit (whole function, whole
prose paragraph, whole config block), or did the split land mid-thought?

In [4]:
for r in samples:
    paragraphs = split_into_paragraphs(r["content"], r["category"])
    print("=" * 100)
    print(f"[{r['category']}] {r['uri']}")
    print(f"{len(r['content'])} chars -> {len(paragraphs)} paragraphs, "
          f"sizes {sorted(len(p) for p, _, _ in paragraphs)}")
    for i, (text, start, end) in enumerate(paragraphs):
        print(f"\n----- p{i}  ({len(text)} chars, ~{len(text) // 4} tokens, chars {start}-{end}) -----")
        print(text)
    print()

[code] git://github.com/branow/webphone/server/src/test/java/com/scisbo/webphone/services/ContactServiceTest.java#chunk-0@c6feb30927daddb0a18847819f083bf36ac30058
10240 chars -> 16 paragraphs, sizes [227, 435, 513, 530, 542, 620, 679, 689, 705, 706, 727, 752, 755, 757, 767, 768]

----- p0  (435 chars, ~108 tokens, chars 0-437) -----
package com.scisbo.webphone.services;
import static org.junit.jupiter.api.Assertions.assertEquals;
import static org.junit.jupiter.api.Assertions.assertThrows;
import static org.mockito.ArgumentMatchers.any;
import static org.mockito.Mockito.never;
import static org.mockito.Mockito.verify;
import static org.mockito.Mockito.when;
import java.util.List;
import java.util.Optional;
import java.util.UUID;
import java.util.stream.Stream;

----- p1  (542 chars, ~135 tokens, chars 439-981) -----
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.extension.ExtendWith;
import org.mockito.junit.jupiter.MockitoExtension;
import org.springframework.beans.fa

## Size distribution over the whole corpus (not just the samples)

Same splitter over every chunk — how many paragraphs, and how many land outside the
100-200-token target band (tiny fragments are the classic recursive-splitter failure:
a lone header or blank-line-separated one-liner becomes its own 'paragraph').

In [5]:
import statistics

token_sizes_by_cat = defaultdict(list)
para_counts = []
for r in rows:
    paragraphs = split_into_paragraphs(r["content"], r["category"])
    para_counts.append(len(paragraphs))
    token_sizes_by_cat[r["category"]] += [len(p) // 4 for p, _, _ in paragraphs]

total = sum(len(v) for v in token_sizes_by_cat.values())
print(f"{len(rows)} chunks -> {total} paragraphs "
      f"(median {statistics.median(para_counts):.0f}/chunk, max {max(para_counts)})\n")
header = f"{'category':<6} {'paras':>6} {'p10':>5} {'median':>7} {'p90':>5} {'max':>6} {'<50 tok':>8} {'50-250':>7} {'>250':>5}"
print(header); print("-" * len(header))
for cat, sizes in sorted(token_sizes_by_cat.items()):
    q = statistics.quantiles(sizes, n=10)
    tiny = sum(s < 50 for s in sizes); band = sum(50 <= s <= 250 for s in sizes)
    print(f"{cat:<6} {len(sizes):>6} {q[0]:>5.0f} {statistics.median(sizes):>7.0f} {q[8]:>5.0f} "
          f"{max(sizes):>6} {100 * tiny / len(sizes):>7.0f}% {100 * band / len(sizes):>6.0f}% "
          f"{100 * (len(sizes) - tiny - band) / len(sizes):>4.0f}%")

510 chunks -> 1829 paragraphs (median 2/chunk, max 17)

category  paras   p10  median   p90    max  <50 tok  50-250  >250
-----------------------------------------------------------------
code     1436    75     158   188    192       4%     96%    0%
data      106    29     119   127    128      17%     83%    0%
docs      287    59     170   189    192       7%     93%    0%


## Observations

(To fill in after eyeball review — kept in this notebook so the evidence and the verdict
live together. The open decision: keep delimiter recursion with tuned settings, or move to
structure-aware boundaries — AST/tree-sitter for code, heading/paragraph trees for docs.)